In [14]:
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

# Load Dataset

DATA_DIR = "donateacry_corpus"

records = []

# Loop through each category folder
for category in os.listdir(DATA_DIR):

    # Create the path of the category folder
    cat_folder = os.path.join(DATA_DIR, category)
    
    if os.path.isdir(cat_folder): # Check if the path is a folder
        wav_files = glob.glob(os.path.join(cat_folder, "*.wav")) # Get all WAV audio files inside the folder
        
        for file_path in wav_files: 
            records.append({"file_path": file_path, "label": category}) # Store the file path and its label

df = pd.DataFrame(records) # Convert the records list into a DataFrame

print(df["label"].value_counts()) # Display the number of samples in each class

# Split dataset into training and testing
train_df, test_df = train_test_split( df, test_size=0.20, random_state=42, stratify=df["label"] )

print("\n(Train Set: 80%) ")
print(train_df["label"].value_counts())

print("\n(Test Set: 20%)")
print(test_df["label"].value_counts())

label
hungry        382
discomfort     27
tired          24
Name: count, dtype: int64

(Train Set: 80%) 
label
hungry        305
discomfort     22
tired          19
Name: count, dtype: int64

(Test Set: 20%)
label
hungry        77
discomfort     5
tired          5
Name: count, dtype: int64


## Features Extraction ##

In [87]:
import numpy as np 
import librosa 

def extract_whole_features(y, sr): 
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) 
    mfcc_mean = np.mean(mfcc, axis=1) 
    mfcc_std = np.std(mfcc, axis=1) 
     
    rms_mean = np.mean(librosa.feature.rms(y=y)) 
     
    zcr_mean = np.mean(librosa.feature.zero_crossing_rate(y=y)) 

    duration = np.array([len(y) / sr])

    f0 = librosa.yin(y, fmin=350, fmax=550, sr=sr)
    f0 = f0[np.isfinite(f0)]  # Remove invalid F0 values
    if len(f0) > 0:
        f0_mean = np.mean(f0)
        f0_std = np.std(f0)
        f0_min = np.min(f0)
        f0_max = np.max(f0)
    else:
        f0_mean = 0
        f0_std = 0
        f0_min = 0
        f0_max = 0
    
    return np.hstack([mfcc_mean, mfcc_std, rms_mean, duration, zcr_mean, f0_mean, f0_std, f0_min, f0_max])

 
def extract_features(y, sr): 
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) 
    mfcc_mean = np.mean(mfcc, axis=1) 
    mfcc_std = np.std(mfcc, axis=1)          

    return np.hstack([mfcc_mean, mfcc_std])

## Audio Data Augmentation ##

In [88]:
import numpy as np
import librosa
import pandas as pd
np.random.seed(42)

# make audo augmentationnn
def augment_audio(y, sr):
    augmented_samples = []
    
    # Pitch shifting 
    augmented_samples.append(librosa.effects.pitch_shift(y=y, sr=sr, n_steps=np.random.uniform(-2, 2)))
    augmented_samples.append(librosa.effects.pitch_shift(y=y, sr=sr, n_steps=np.random.uniform(-2, 2)))

    # Time stretching 
    augmented_samples.append(librosa.effects.time_stretch(y=y, rate=np.random.uniform(0.9, 1.1)))
    augmented_samples.append(librosa.effects.time_stretch(y=y, rate=np.random.uniform(0.9, 1.1)))
    
    # added  Gaussian white noise
    noise = np.random.randn(len(y))
    noise_factor = 0.005
    augmented_samples.append(y + (noise_factor * noise))
    
    return augmented_samples



In [89]:
# 2. Extract features for training set with augmentation on minority classes
X_train = []
y_train = []

for _, row in train_df.iterrows():
    y, sr = librosa.load(row['file_path'], sr=16000)
    
    # Extract features from original audio
    X_train.append(extract_features(y, sr))

    y_train.append(row['label'])

    
    # Augment minority classes only 
    if row['label'] != 'hungry':

        for aug_y in augment_audio(y, sr):

            X_train.append(extract_features(aug_y, sr))

            y_train.append(row['label'])

X_train = np.array(X_train)

y_train = np.array(y_train)



# 3. Extract features for test set without augmentation
X_test = []

y_test = []

print("Processing test set ")

for _, row in test_df.iterrows():
    y, sr = librosa.load(row['file_path'], sr=16000)
    X_test.append(extract_features(y, sr))
    y_test.append(row['label'])

X_test = np.array(X_test)
y_test = np.array(y_test)


# 4. Print shapes and class balances
print("\nSummary ")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

print("\nTraining set distribution after augmentation:")
print(pd.Series(y_train).value_counts())

print("\nTest set distribution:")
print(pd.Series(y_test).value_counts())

c:\Users\Catri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\librosa\effects.py:448: FutureWarning: The `hop_length` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
c:\Users\Catri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\librosa\effects.py:448: FutureWarning: The `n_fft` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(


Processing test set 

Summary 
X_train shape: (551, 26)
y_train shape: (551,)
X_test shape:  (87, 26)
y_test shape:  (87,)

Training set distribution after augmentation:
hungry        305
discomfort    132
tired         114
Name: count, dtype: int64

Test set distribution:
hungry        77
discomfort     5
tired          5
Name: count, dtype: int64


## **Random Forest**

In [90]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

model = RandomForestClassifier(
    criterion="entropy",
    n_estimators=500,
    random_state=None,
    n_jobs=-1
)
"""n_estimators=200,
    criterion="gini",
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    class_weight="balanced",
    bootstrap=True,
    n_jobs=-1
"""

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  discomfort       0.25      0.20      0.22         5
      hungry       0.90      0.97      0.94        77
       tired       0.00      0.00      0.00         5

    accuracy                           0.87        87
   macro avg       0.38      0.39      0.39        87
weighted avg       0.81      0.87      0.84        87

[[ 1  4  0]
 [ 2 75  0]
 [ 1  4  0]]


c:\Users\Catri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Catri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Catri\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave